# Peters Dome: Sentinel-1 RTC before and after the ice-rock avalanche

On 2023-09-14 an ice-rock avalanche of about 6.1 × 10⁶ m³ fell from Peters Dome in Denali National Park and Preserve, Alaska, described by Toney et al. (2026), [*JGR Earth Surface*, doi:10.1029/2025JF008577](https://agupubs.onlinelibrary.wiley.com/doi/10.1029/2025JF008577). The source area and its outlines are from the accompanying USGS data release, [doi:10.5066/P13WIS6X](https://doi.org/10.5066/P13WIS6X): `source` (from DEM differencing) and `total`, the source plus the deposit (from SAR differencing).

This notebook is a starting point for working with the Sentinel-1 RTC passes that `gaia stage` writes for [peters-dome.yaml](peters-dome.yaml): it lists the passes, compares the last pass before the event with the first one after it on the same track, zooms in on the mapped outlines, and plots backscatter inside them through time.

Stage the data first, from the repository root:

```bash
pixi run gaia stage --config examples/peters-dome.yaml --dry-run       # what will be read
pixi run gaia stage --config examples/peters-dome.yaml -o examples/scratch/peters-dome
```

or into the public scratch bucket (Kopah, with write credentials in `AWS_PROFILE`):

```bash
AWS_PROFILE=kopah pixi run gaia stage --config examples/peters-dome.yaml -o s3://gaia-scratch/gaia-cli/peters-dome
```

Values are gamma0 as **linear power**; `10 log10` gives dB.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rioxarray  # noqa: F401  registers the .rio accessor
import xarray as xr
from obstore.store import HTTPStore
from pyproj import Transformer
from zarr.storage import ObjectStore

from gaia_cli import io

# A local folder from `gaia stage -o`, an s3:// URL (needs AWS_PROFILE), or the public
# https:// URL of a store in s3://gaia-scratch, read without credentials
STORE = "scratch/peters-dome/peters-dome.zarr"
# STORE = "https://s3.kopah.uw.edu/gaia-scratch/gaia-cli/peters-dome/peters-dome.zarr"

EVENT = pd.Timestamp("2023-09-14T02:06:49")  # USGS ComCat origin time, UTC
SOURCE = (-151.1871, 63.1423)  # lon, lat of the source area (USGS data release)
# The release's outlines (a zipped shapefile): `source` and `total` (source + deposit)
OUTLINES = (
    "/vsizip/{/vsicurl/https://www.sciencebase.gov/catalog/file/get/6927572fd4be02716f722db3"
    "?f=__disk__e3%2F36%2F7f%2Fe3367f546f1d09d4e014d8d8968bb6cb7ac6303f}"
    "/dem_diff_source_sar_diff_total/dem_diff_source_sar_diff_total.shp"
)


def open_store(path):
    if path.startswith("https://"):
        return xr.open_datatree(ObjectStore(HTTPStore.from_url(path), read_only=True), engine="zarr")
    return io.open_datatree(path)


def db(power):
    return 10 * np.log10(power)


s1 = open_store(STORE)["rasters/sentinel1-rtc"].to_dataset()
to_utm = Transformer.from_crs("EPSG:4326", s1.rio.crs, always_xy=True)
x0, y0 = to_utm.transform(*SOURCE)
outlines = gpd.read_file(OUTLINES).to_crs(s1.rio.crs).set_index("name")
s1

## The passes

Each time step is one satellite pass; its platform and orbit are coordinates along `time`.

In [ ]:
scenes = s1.drop_dims(["x", "y"]).reset_coords().drop_vars("spatial_ref").to_dataframe()
scenes["days_from_event"] = ((scenes.index - EVENT) / pd.Timedelta(days=1)).round(1)
scenes

## Before and after, on the same track

A change map needs two passes with the same viewing geometry, so pair the last pass before the event with the first pass after it **on each relative orbit**. Power is averaged over 5 × 5 cells (50 m) before the log ratio, to reduce speckle. The magenta dot is the source area, the solid outline the source and the dashed outline the source plus deposit. (Choosing these passes at staging time is the next step for `gaia stage`: `pre`/`post` windows with `match: relative_orbit`.)

In [ ]:
pairs = {}
for track, ds in s1.groupby("relative_orbit"):
    before, after = ds.sel(time=slice(None, EVENT)), ds.sel(time=slice(EVENT, None))
    if before.sizes["time"] and after.sizes["time"]:
        pairs[int(track)] = (before.isel(time=-1), after.isel(time=0))
{track: [str(p.time.values)[:16] for p in pair] for track, pair in pairs.items()}

In [ ]:
def before_after(pre, post, title):
    """Before, after and the change in dB, with the source area and outlines in magenta."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharex=True, sharey=True, layout="constrained")
    for ax, scene, label in ((axes[0], pre, "before"), (axes[1], post, "after")):
        db(scene.vv).plot.imshow(ax=ax, cmap="gray", vmin=-25, vmax=5, cbar_kwargs={"label": "γ⁰ VV (dB)"})
        ax.set_title(f"{label}: {str(scene.time.values)[:10]}")
    (db(post.vv) - db(pre.vv)).plot.imshow(
        ax=axes[2], cmap="RdBu_r", vmin=-6, vmax=6, cbar_kwargs={"label": "after − before (dB)"}
    )
    axes[2].set_title("change")
    for ax in axes:
        ax.plot(x0, y0, "o", color="magenta", ms=6, label="source area")
        outlines.loc[["source"]].boundary.plot(ax=ax, color="magenta", lw=1)
        outlines.loc[["total"]].boundary.plot(ax=ax, color="magenta", lw=1, ls="--")
        ax.set_aspect("equal")
    axes[0].legend(loc="lower left")
    fig.suptitle(title)


for track, pair in pairs.items():
    pre, post = (p.coarsen(x=5, y=5, boundary="trim").mean() for p in pair)  # 50 m
    before_after(pre, post, f"Sentinel-1 RTC VV at 50 m, relative orbit {track} ({pair[0].orbit_state.item()})")

## Zoom on the mapped outlines

The release's source + deposit outline, padded by 1 km, with power averaged over 2 × 2 cells (20 m).

In [ ]:
track, (pre, post) = next(iter(pairs.items()))  # the first track with a pair
west, south, east, north = outlines.total_bounds
zoom = dict(x=slice(west - 1000, east + 1000), y=slice(north + 1000, south - 1000))  # y decreases
pre_z, post_z = (p.sel(**zoom).coarsen(x=2, y=2, boundary="trim").mean() for p in (pre, post))
before_after(pre_z, post_z, f"Sentinel-1 RTC VV at 20 m around the outlines, relative orbit {track}")

## Backscatter inside the outlines through time

The median of each pass inside the source and the source + deposit outlines. The `total` outline was mapped from SAR differencing, so its change is partly by construction.

In [ ]:
medians = {
    name: db(s1[["vv", "vh"]].rio.clip([outlines.geometry[name]]).median(["x", "y"])).to_dataframe()[["vv", "vh"]]
    for name in ("source", "total")
}
series = pd.concat(medians, axis=1)

fig, ax = plt.subplots(figsize=(9, 3.5))
for (name, pol), values in series.items():
    ax.plot(values.index, values, "o-" if name == "total" else "s--", label=f"{pol.upper()}, {name}")
ax.axvline(EVENT, color="k", ls="--", lw=1, label="event")
ax.set_ylabel("median γ⁰ (dB)")
ax.legend(ncols=2, fontsize="small")
series.round(2)